In [2]:
# , N-BEATS is especially interesting for univariate forecasting
# Yes. N-BEATS (Neural Basis Expansion Analysis for Interpretable Time Series Forecasting) is another deep-learning model you
# can use for time-series forecasting

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [5]:
pip install nbeats-keras

  Using cached nbeats_keras-1.8.0-py3-none-any.whl.metadata (5.0 kB)
INFO: pip is looking at multiple versions of tensorflow to determine which version is compatible with other requirements. This could take a while.
  Using cached tensorflow-2.20.0-cp310-cp310-win_amd64.whl.metadata (4.6 kB)
  Using cached tensorflow-2.19.1-cp310-cp310-win_amd64.whl.metadata (4.1 kB)
  Using cached tensorflow-2.19.0-cp310-cp310-win_amd64.whl.metadata (4.1 kB)
  Using cached tensorflow-2.18.1-cp310-cp310-win_amd64.whl.metadata (4.1 kB)
  Using cached tensorflow-2.18.0-cp310-cp310-win_amd64.whl.metadata (3.3 kB)
  Using cached tensorflow_intel-2.18.0-cp310-cp310-win_amd64.whl.metadata (4.9 kB)
INFO: pip is looking at multiple versions of tensorflow-intel to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of tensorflow to determine which version is compatible with other requirements. This could take a while.
  Using ca

ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\Mcc\\AppData\\Local\\Programs\\Python\\Python310\\Lib\\site-packages\\tensorflow\\compiler\\tf2tensorrt\\_pywrap_py_utils.pyd'
Consider using the `--user` option or check the permissions.



In [6]:
from nbeats_keras.model import NBeatsNet

ModuleNotFoundError: No module named 'nbeats_keras'

In [ ]:

df = pd.read_csv("AirPassengers.csv")

df["Month"] = pd.to_datetime(df["Month"])

df = df.set_index("Month")

print(df.head())

In [ ]:
plt.figure(figsize=(14, 5))

plt.plot(
    data.index,
    data["#Passengers"]
)

plt.xlabel("Date")
plt.ylabel("Passengers")
plt.title("AirPassengers")
plt.grid()
plt.show()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

In [ ]:
window = 24
X = []
y = []

for i in range(window, len(scaled_data)):
    
    X.append(
        scaled_data[i-window:i, 0]
    )
    
    y.append(
        scaled_data[i, 0]
    )

X = np.array(X)
y = np.array(y)

print(X.shape)
print(y.shape)

In [ ]:
train_end = int(len(X) * 0.70)
val_end = int(len(X) * 0.85)

X_train = X[:train_end]
y_train = y[:train_end]

X_val = X[train_end:val_end]
y_val = y[train_end:val_end]

X_test = X[val_end:]
y_test = y[val_end:]

In [ ]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

In [ ]:
dates = data.index[window:]
dates_train = dates[:train_end]

dates_val = dates[train_end:val_end]

dates_test = dates[val_end:]

In [ ]:
print(len(X_train), len(dates_train))
print(len(X_val), len(dates_val))
print(len(X_test), len(dates_test))

In [ ]:
train_end = int(len(data) * 0.70)
val_end = int(len(data) * 0.85)

train_data = data.iloc[:train_end]
val_data = data.iloc[train_end:val_end]
test_data = data.iloc[val_end:]

In [ ]:
print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))

In [ ]:
print(len(X_train), len(dates_train))
print(len(X_val), len(dates_val))
print(len(X_test), len(dates_test))

In [ ]:
from nbeats_keras.model import NBeatsNet
model = NBeatsNet(
    backcast_length=24,
    forecast_length=1,

    stack_types=(
        NBeatsNet.TREND_BLOCK,
        NBeatsNet.SEASONALITY_BLOCK
    ),

    nb_blocks_per_stack=2,

    thetas_dim=(4, 8),

    share_weights_in_stack=True,

    hidden_layer_units=64


In [ ]:
model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="mse"
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=15,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

In [ ]:
history = model.fit(
    X_train,
    y_train,
    epochs=200,
    batch_size=8,
    validation_data=(X_val, y_val),
    shuffle=False,
    callbacks=[
        early_stop,
        reduce_lr
    ]
)

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    history.history["loss"],
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("N-BEATS Training vs Validation Loss")

plt.legend()
plt.grid()

plt.show()

In [ ]:
prediction = model.predict(X_test)

In [ ]:
print(type(prediction))
print(np.shape(prediction))

In [ ]:
y_test_scaled = y_test.reshape(-1, 1)
y_pred_scaled = np.array(y_pred).reshape(-1, 1)

In [ ]:
y_test_actual = scaler.inverse_transform(
    y_test_scaled
)

y_pred_actual = scaler.inverse_transform(
    y_pred_scaled
)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

mae = mean_absolute_error(
    y_test_actual,
    y_pred_actual
)

rmse = np.sqrt(
    mean_squared_error(
        y_test_actual,
        y_pred_actual
    )
)

print("MAE:", mae)
print("RMSE:", rmse)

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    dates_test,
    y_test_actual.flatten(),
    label="Actual",
    color="blue",
    linewidth=2
)

plt.plot(
    dates_test,
    y_pred_actual.flatten(),
    label="N-BEATS Predicted",
    color="red",
    linewidth=2
)

plt.xlabel("Date")
plt.ylabel("Passengers")

plt.title(
    "N-BEATS: Actual vs Predicted"
)

plt.legend()
plt.grid()

plt.show()

In [ ]:
last_sequence = scaled_data[-24:, 0]
X_future = last_sequence.reshape(1, 24)
future_prediction = model.predict(X_future)
future_prediction = future_prediction[1]

In [ ]:
future_prediction = np.array(
    future_prediction
).reshape(-1, 1)

future_prediction = scaler.inverse_transform(
    future_prediction
)

In [ ]:
next_month = data.index[-1] + pd.DateOffset(months=1)

In [ ]:
future_df = pd.DataFrame({
    "Month": [next_month],
    "Predicted_Passengers": future_prediction.flatten()
})

print(future_df)

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    data.index,
    data["#Passengers"],
    label="Historical",
    color="blue"
)

plt.plot(
    future_df["Month"],
    future_df["Predicted_Passengers"],
    color="red",
    marker="o",
    markersize=8,
    label="N-BEATS Forecast"
)

plt.xlabel("Date")
plt.ylabel("Passengers")

plt.title(
    "N-BEATS Future Forecast"
)

plt.legend()
plt.grid()

plt.show()